# 4. MLflow: трекинг экспериментов и реестр моделей

В ноутбуке 02 результаты сравнения моделей остались только в выводе ячеек: перезапустишь — и старые цифры пропадут. **MLflow** решает это:

* **Tracking** — каждый запуск обучения (*run*) сохраняется вместе с гиперпараметрами, метриками и файлами (*артефактами*). Запуски можно сравнивать в веб-интерфейсе.
* **Model Registry** — «каталог» готовых моделей: у модели есть имя, номера версий и алиасы (например, `champion` — та версия, которая сейчас работает в API).

Всё хранится локально: база `mlflow.db` (запуски и реестр) и папка `mlruns/` (файлы моделей).

**Посмотреть результаты в браузере** — в терминале, в папке проекта:
```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
и открыть http://127.0.0.1:5000

In [1]:
import os
import warnings
from pathlib import Path

# MLflow печатает много служебных подсказок и полос загрузки — для учебного проекта убираем шум
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="Hint: Inferred schema contains integer")

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

EXPERIMENT = "us-election-prediction"
MODEL_NAME = "us-election-predictor"

mlflow.set_tracking_uri("sqlite:///../mlflow.db")  # база в корне проекта
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=Path("../mlruns").resolve().as_uri())
experiment = mlflow.set_experiment(EXPERIMENT)
print("эксперимент:", experiment.name, "| id:", experiment.experiment_id)

2026/11/27 22:13:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/27 22:13:57 INFO mlflow.store.db.utils: Updating database tables


эксперимент: us-election-prediction | id: 1


## 4.1 Данные и честная проверка

Берём таблицу признаков из ноутбука 02 и ту же проверку на «будущих» выборах 1996–2024. Теперь считаем четыре метрики:

| метрика | что показывает | лучше, когда |
|---|---|---|
| `accuracy` | доля штатов с угаданным победителем | больше |
| `f1` | баланс точности и полноты; *macro* — среднее по обоим классам, чтобы не зависеть от того, какой класс считать «положительным» | больше |
| `log_loss` | штраф за уверенные ошибки — честность вероятностей | меньше |
| `roc_auc` | насколько хорошо модель ранжирует штаты от «точно удержит» до «точно потеряет» | больше |

In [2]:
FEATURES = ["inc_prev_share", "inc_prev_lean", "inc_running", "inc_long_tenure", "inc_home_state"]
TARGET = "inc_win"
TEST_YEARS = [1996, 2000, 2004, 2008, 2012, 2016, 2020, 2024]

data = pd.read_csv("../data/processed/state_features.csv").dropna(subset=[TARGET])
data[TARGET] = data[TARGET].astype(int)


def backtest(model):
    """Для каждого года T: учимся на выборах до T, предсказываем T."""
    parts = []
    for year in TEST_YEARS:
        train, test = data[data.year < year], data[data.year == year]
        fitted = clone(model).fit(train[FEATURES], train[TARGET])
        parts.append(test.assign(proba=fitted.predict_proba(test[FEATURES])[:, 1]))
    return pd.concat(parts)


def metrics(pred):
    y, proba, label = pred[TARGET], pred.proba, pred.proba > 0.5
    return {
        "accuracy": accuracy_score(y, label),
        "f1": f1_score(y, label, average="macro"),
        "log_loss": log_loss(y, proba.clip(0.01, 0.99)),
        "roc_auc": roc_auc_score(y, proba),
    }

## 4.2 Кандидаты: модели и гиперпараметры

В ноутбуке 02 у каждой модели был один набор настроек. Здесь попробуем несколько **гиперпараметров** — настроек, которые задаём до обучения:

* `C` у логистической регрессии — сила регуляризации: чем меньше `C`, тем сильнее модель «прижимает» коэффициенты к нулю;
* `min_samples_leaf` у случайного леса — минимум строк в листе дерева: чем больше, тем проще деревья;
* `max_depth` у бустинга — глубина деревьев (1 = «пень» с одним вопросом).

In [3]:
def build(model_type, params):
    if model_type == "logistic_regression":
        return make_pipeline(StandardScaler(), LogisticRegression(**params))
    if model_type == "random_forest":
        return RandomForestClassifier(n_estimators=300, random_state=42, **params)
    return GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, random_state=42, **params)


candidates = {
    "logreg_C0.1": ("logistic_regression", {"C": 0.1}),
    "logreg_C1": ("logistic_regression", {"C": 1.0}),
    "logreg_C10": ("logistic_regression", {"C": 10.0}),
    "forest_leaf5": ("random_forest", {"min_samples_leaf": 5}),
    "forest_leaf20": ("random_forest", {"min_samples_leaf": 20}),
    "boosting_depth1": ("gradient_boosting", {"max_depth": 1}),
    "boosting_depth2": ("gradient_boosting", {"max_depth": 2}),
}

## 4.3 Логируем каждый запуск в MLflow

Все кандидаты — «дочерние» запуски одного родительского `model_selection`, чтобы в интерфейсе они были сгруппированы. Для каждого сохраняем:

* `log_params` — тип модели и гиперпараметры;
* `log_metrics` — четыре метрики;
* `log_metric(..., step=год)` — точность по годам: MLflow нарисует из этого график.

Бейзлайн («как в прошлый раз») тоже логируем — чтобы в интерфейсе было с чем сравнивать.

In [4]:
def log_run(run_name, params, pred):
    with mlflow.start_run(run_name=run_name, nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics(pred))
        for year, acc in (pred.proba > 0.5).eq(pred[TARGET]).groupby(pred.year).mean().items():
            mlflow.log_metric("accuracy_by_year", acc, step=year)


with mlflow.start_run(run_name="model_selection") as parent:
    mlflow.log_params({"features": ", ".join(FEATURES), "test_years": "1996-2024",
                       "target": "incumbent party wins state"})

    test = data[data.year.isin(TEST_YEARS)]
    log_run("baseline", {"model_type": "baseline"},
            test.assign(proba=(test.inc_prev_share > 0.5).astype(float)))

    for run_name, (model_type, params) in candidates.items():
        log_run(run_name, {"model_type": model_type, **params}, backtest(build(model_type, params)))

print("родительский запуск:", parent.info.run_id)

родительский запуск: f322e86c921343168451c0726aee525d


## 4.4 Выбираем лучшую модель — запросом к MLflow

Теперь результаты не нужно переписывать из вывода: их можно получить из MLflow как таблицу.

In [5]:
runs = mlflow.search_runs(
    filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'",
    order_by=["metrics.log_loss ASC"],
)
columns = ["tags.mlflow.runName", "params.model_type", "metrics.accuracy", "metrics.f1",
           "metrics.log_loss", "metrics.roc_auc"]
runs[columns].round(3)

,tags.mlflow.runName,params.model_type,metrics.accuracy,metrics.f1,metrics.log_loss,metrics.roc_auc
0,boosting_depth1,gradient_boosting,0.909,0.909,0.245,0.976
1,logreg_C1,logistic_regression,0.902,0.902,0.247,0.970
2,forest_leaf5,random_forest,0.904,0.904,0.265,0.969
3,boosting_depth2,gradient_boosting,0.887,0.887,0.272,0.961
4,logreg_C0.1,logistic_regression,0.897,0.897,0.286,0.972
5,forest_leaf20,random_forest,0.904,0.904,0.286,0.972
6,logreg_C10,logistic_regression,0.843,0.841,0.359,0.939
7,baseline,baseline,0.885,0.884,0.539,0.887


Формально лучший log loss — у бустинга с глубиной 1, но разница с логистической регрессией — тысячные доли: на 408 прогнозах это шум. Правило, которое часто используют на практике: **если модели почти равны, берём самую простую** — её легче объяснить (коэффициенты видны, см. ноутбук 02) и меньше риск переобучения.

In [6]:
TOLERANCE = 0.01  # разница в log loss меньше этой — считаем модели равными
SIMPLICITY = {"logistic_regression": 0, "random_forest": 1, "gradient_boosting": 2}

models_only = runs[runs["params.model_type"] != "baseline"]
close = models_only[models_only["metrics.log_loss"] <= models_only["metrics.log_loss"].min() + TOLERANCE]
best = (close.assign(simplicity=close["params.model_type"].map(SIMPLICITY))
        .sort_values(["simplicity", "metrics.log_loss"]).iloc[0])

best_name = best["tags.mlflow.runName"]
best_type, best_params = candidates[best_name]
print("почти равны лучшей:", close["tags.mlflow.runName"].tolist())
print("выбрана:", best_name, best_params)

почти равны лучшей: ['boosting_depth1', 'logreg_C1']
выбрана: logreg_C1 {'C': 1.0}


## 4.5 Финальная модель и Model Registry

Обучаем выбранную модель на **всех** выборах 1980–2024 и сохраняем:

1. файл `model.joblib` — его загружает наш FastAPI; кладём его и в артефакты запуска;
2. модель в формате MLflow (`log_model`) — с примером входных данных и схемой признаков (*signature*);
3. **регистрируем** её в реестре под именем `us-election-predictor`. Каждый запуск этого ноутбука создаёт новую **версию**: 1, 2, 3…

In [7]:
final_model = build(best_type, best_params).fit(data[FEATURES], data[TARGET])
joblib.dump(final_model, "../model.joblib")

with mlflow.start_run(run_name="final_model") as final_run:
    mlflow.log_params({"model_type": best_type, **best_params, "train_years": "1980-2024",
                       "features": ", ".join(FEATURES)})
    mlflow.log_metrics({name: best[f"metrics.{name}"] for name in ["accuracy", "f1", "log_loss", "roc_auc"]})
    mlflow.log_artifact("../model.joblib")  # сам файл модели, который использует API
    model_info = mlflow.sklearn.log_model(
        final_model,
        name="model",
        registered_model_name=MODEL_NAME,
        input_example=data[FEATURES].head(3).astype(float),
    )

version = model_info.registered_model_version
print(f"зарегистрирована модель {MODEL_NAME}, версия {version}")

зарегистрирована модель us-election-predictor, версия 1


Successfully registered model 'us-election-predictor'.
Created version '1' of model 'us-election-predictor'.


Добавим описание и алиас `champion`. Алиас — это «ярлык», который можно перевесить на другую версию: API или frontend могут просить «модель champion», не зная её номера.

In [8]:
client = MlflowClient()
client.update_registered_model(
    MODEL_NAME, description="Победитель президентских выборов США в штате: удержит ли его партия президента.")
client.update_model_version(
    MODEL_NAME, version,
    description=f"{best_name}: accuracy {best['metrics.accuracy']:.3f}, log loss {best['metrics.log_loss']:.3f} "
                f"(проверка на выборах 1996–2024), обучена на 1980–2024")
client.set_registered_model_alias(MODEL_NAME, "champion", version)

versions = [client.get_model_version(MODEL_NAME, v.version)
            for v in client.search_model_versions(f"name = '{MODEL_NAME}'")]
pd.DataFrame([{"version": v.version, "aliases": v.aliases, "run_id": v.run_id, "description": v.description}
              for v in versions]).sort_values("version")

,version,aliases,run_id,description
0,1,[champion],a4c5b4b4cfcd4359b2938a4c1ac88624,"logreg_C1: accuracy 0.902, log loss 0.247 (про..."


## 4.6 Проверка: загружаем модель из реестра

Загружаем версию с алиасом `champion` по адресу `models:/us-election-predictor@champion` и сравниваем с `model.joblib`, который использует API. Прогнозы должны совпасть.

In [9]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
api_model = joblib.load("../model.joblib")

example = data[data.year == 2024].set_index("state").loc[["PA", "WI", "TX", "CA"], FEATURES]
pd.DataFrame({
    "из реестра (champion)": champion.predict_proba(example)[:, 1],
    "model.joblib (API)": api_model.predict_proba(example)[:, 1],
}, index=example.index).round(3)

,из реестра (champion),model.joblib (API)
state,,
PA,0.540,0.540
WI,0.512,0.512
TX,0.226,0.226
CA,1.000,1.000


## Итоги

| что требуется | где сделано |
|---|---|
| эксперимент MLflow | `us-election-prediction` (4.1) |
| параметры модели | `log_params`: тип модели, `C`, `min_samples_leaf`, `max_depth` (4.3) |
| метрики | accuracy, F1, log loss, ROC AUC + точность по годам (4.3) |
| артефакты | `model.joblib` и модель в формате MLflow (4.5) |
| реестр моделей | `us-election-predictor`, номер версии + алиас `champion` (4.5) |

**Что посмотреть в интерфейсе** (`mlflow ui --backend-store-uri sqlite:///mlflow.db`):
* *Experiments → us-election-prediction* — запуски; раскрой `model_selection`, отметь дочерние запуски и нажми **Compare**: таблица и графики метрик;
* в запуске `final_model` → *Artifacts* — файл `model.joblib` и папка модели;
* *Models → us-election-predictor* — версии и алиас `champion`.